In [ ]:
import sys
from pathlib import Path

root_dir = Path().absolute()

if root_dir.parts[-1:] == ('airquality',):
    root_dir = Path(*root_dir.parts[:-1])

if root_dir.parts[-1:] == ('notebooks',):
    root_dir = Path(*root_dir.parts[:-1])

root_dir = str(root_dir) 
print(f"Root dir: {root_dir}")

if root_dir not in sys.path:
    sys.path.append(root_dir)
    print(f"Added the following directory to the PYTHONPATH: {root_dir}")

from mlfs import config

settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

In [ ]:
import os
from mlfs.airquality import util
import hopsworks
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from xgboost import plot_importance
from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime, timedelta

In [ ]:
project = hopsworks.login(engine="python")
AQICN_API_KEY = settings.AQICN_API_KEY.get_secret_value() 
aqicn_url = "https://api.waqi.info/feed/@13983"
country = "Sweden"
city = "Stockholm"
street = "Sollentuna Ekmans Väg 11"
#latitude, longitude = util.get_city_coordinates(city)
latitude = "59.314017450051836"
longitude = "18.075630285191593"
secrets = hopsworks.get_secrets_api()
secret = secrets.get_secret("AQICN_API_KEY")
if secret is not None:
    secret.delete()
    print("Replacing existing AQICN_API_KEY")

secrets.create_secret("AQICN_API_KEY", AQICN_API_KEY)
today = datetime.today()

try:
    aq_today_df = util.get_pm25(aqicn_url, country, city, street, today, AQICN_API_KEY)
except hopsworks.RestAPIError:
    print("It looks like the AQICN_API_KEY doesn't work for your sensor. Is the API key correct? Is the sensor URL correct?")

aq_today_df.head()

Fetch the feature groups

In [ ]:
fs = project.get_feature_store()
air_quality_fg = fs.get_feature_group(
    name='air_quality',
    version=1,
)
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)
air_quality_fg

Read air quality features into a Pandas DataFrame

In [ ]:
df_aq_lag = air_quality_fg.read()
df_aq_lag

In [ ]:
# Sort and interpolate missing dates.
df_aq_lag = df_aq_lag.set_index("date").sort_index()
full_idx = pd.date_range(
    start=df_aq_lag.index.min(),
    end=df_aq_lag.index.max(),
    freq="D"
)
df_aq_lag = df_aq_lag.reindex(full_idx)
df_aq_lag.index.name = "date"
df_aq_lag["pm25"] = df_aq_lag["pm25"].interpolate(method="time")
df_aq_lag = df_aq_lag.reset_index()

# Add supplementary info
df_aq_lag['country']=country
df_aq_lag['city']=city
df_aq_lag['street']=street
df_aq_lag['url']=aqicn_url

plt.plot(df_aq_lag["date"], df_aq_lag["pm25"])
plt.show()
df_aq_lag

Add features for previous 1-3 days pm25 readings

In [ ]:
for k in [1, 2, 3]:
    df_aq_lag[f"pm25_lag{k}"] = df_aq_lag["pm25"].shift(k)

df_aq_lag.dropna(inplace=True)
df_aq_lag.info()
df_aq_lag

Add data expectation

In [ ]:
import great_expectations as ge

aq_expectation_suite = ge.core.ExpectationSuite(
    expectation_suite_name="aq_expectation_suite"
)

aq_expectation_suite.add_expectation(
    ge.core.ExpectationConfiguration(
        expectation_type="expect_column_min_to_be_between",
        kwargs={
            "column":"pm25",
            "min_value":-0.1,
            "max_value":500.0,
            "strict_min":True
        }
    )
)

Create a new feature group to store lagged air quality dataframe.

In [ ]:
aq_lag_fg = fs.get_or_create_feature_group(
    name='aq_lag',
    description='Lagged air quality for the previous 1-3 days',
    version=1,
    primary_key=['country','city', 'street'],
    event_time="date",
    expectation_suite=aq_expectation_suite
)
aq_lag_fg.insert(df_aq_lag)
aq_lag_fg.update_feature_description("date", "Date of measurement of air quality")
aq_lag_fg.update_feature_description("country", "Country where the air quality was measured (sometimes a city in acqcn.org)")
aq_lag_fg.update_feature_description("city", "City where the air quality was measured")
aq_lag_fg.update_feature_description("street", "Street in the city where the air quality was measured")
aq_lag_fg.update_feature_description("pm25", "Particles less than 2.5 micrometers in diameter (fine particles) pose health risk")
aq_lag_fg.update_feature_description("pm25_lag1", "PM25 reading 1 day prior")
aq_lag_fg.update_feature_description("pm25_lag2", "PM25 reading 2 days prior")
aq_lag_fg.update_feature_description("pm25_lag3", "PM25 reading 3 days prior")

Now, that we've added a new features, we can load our model and retrain it on the new features.

Create a new feature view to train on.

In [ ]:
# Select features for training data.
selected_features = aq_lag_fg.select(['pm25', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'date']).join(weather_fg.select_features(), on=['city'])
feature_view = fs.get_or_create_feature_view(
    name='aq_lag_fv',
    description="weather and lagged 1-3 day air quality features with air quality as the target",
    version=1,
    labels=['pm25'],
    query=selected_features,
)

In [ ]:
start_date_test_data = "2025-05-01"
# Convert string to datetime object
test_start = datetime.strptime(start_date_test_data, "%Y-%m-%d")
X_train, X_test, y_train, y_test = feature_view.train_test_split(
    test_start=test_start
)
X_features = X_train.drop(columns=['date'])
X_test_features = X_test.drop(columns=['date'])

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
# Creating an instance of the XGBoost Regressor
xgb_regressor = XGBRegressor()

# Fitting the XGBoost Regressor to the training data
xgb_regressor.fit(X_features, y_train)

# Predicting target values on the test set
y_pred = xgb_regressor.predict(X_test_features)

# Calculating Mean Squared Error (MSE) using sklearn
mse = mean_squared_error(y_test.iloc[:,0], y_pred)
print("MSE:", mse)

# Calculating R squared using sklearn
r2 = r2_score(y_test.iloc[:,0], y_pred)
print("R squared:", r2)

df = y_test
df['predicted_pm25'] = y_pred
df['date'] = X_test['date']
df = df.sort_values(by=['date'])
df.head(5)

In [ ]:
# Creating a directory for the model artifacts if it doesn't exist
model_dir = "air_quality_model"

if not os.path.exists(model_dir):
    os.mkdir(model_dir)

images_dir = model_dir + "/images"

if not os.path.exists(images_dir):
    os.mkdir(images_dir)

model_path = model_dir + "/new_model.json"
xgb_regressor.save_model(model_path)
file_path = images_dir + "/new_pm25_hindcast.png"
plt = util.plot_air_quality_forecast(city, street, df, file_path, hindcast=True) 
plt.show()
res_dict = { 
        "MSE": str(mse),
        "R squared": str(r2),
    }

In [ ]:
mr = project.get_model_registry()

# Creating a Python model in the model registry named 'new_air_quality_xgboost_model'

aq_model = mr.python.create_model(
    name="air_quality_xgboost_model", 
    metrics= res_dict,
    feature_view=feature_view,
    description="Air Quality (PM2.5) predictor",
)

# Saving the model artifacts to the 'air_quality_model' directory in the model registry
aq_model.save(model_path)